<span style="font-family:monospace;letter-spacing:.14em;color:#857259">BANA · MODULE 3 · STATISTICAL REASONING FOR BUSINESS</span>

# Session 4: Reading the result: intervals, effect size and the Project Charter v2

**Data:** EV purchase intention (Kaggle Playground Series S6E9, a *synthetic* dataset). `ev_m3_sample.csv` is the cleaned 10,000-row class sample from Module 2. `ev_m3_population.csv` has 50,000 rows that we treat as **everybody**, so we can see what sampling does.
**Time:** 60 minutes. **Course outcome:** CO3.

| Block | Min | What you do |
|---|---|---|
| Hook | 5 | p = 0.005 for chargers near home: should marketing act? |
| Concept | 10 | Confidence interval, effect size, many tests, power (instructor) |
| Guided lab | 30 | Intervals, effect sizes, multiple tests, power |
| Team task | 10 | Project Charter v2 and a note to the marketing head |
| Exit ticket | 5 | Three quick checks |

**By the end of this session you can:**
- read a confidence interval in plain English
- tell a *significant* result from an *important* one using an effect size
- explain why testing many columns produces false alarms
- explain why a non-significant result is not proof of 'no effect'

**New commands this session (max 5):** `proportion_confint`, `.confidence_interval()`, Cohen's d (formula), `stats.pearsonr`, Bonferroni (alpha divided by number of tests)

**AI-use (AMBER protocol).** You may use an AI tool to look up *syntax*. The reasoning and interpretation must be your own: you will defend them in the viva.

In [ ]:
# -- Setup: run once ------------------------------------------------------
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
warnings.filterwarnings("ignore")

# House style: Nikhil D Jonathan design system (sand ground, terracotta / lagoon / ochre data colours)
C = {"lagoon": "#25707B", "terracotta": "#B5562F", "ochre": "#E0A231", "moss": "#3F6F4B",
     "earth": "#665643", "ink": "#221B15", "sand": "#F8F3EA", "card": "#FFFDF9", "line": "#E3D6C1"}
sns.set_theme(style="whitegrid", rc={
    "figure.facecolor": C["sand"], "axes.facecolor": C["card"], "axes.edgecolor": "#CDBCA3",
    "grid.color": C["line"], "text.color": C["ink"], "axes.labelcolor": C["ink"],
    "xtick.color": "#4A3E31", "ytick.color": "#4A3E31", "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 100, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.titlelocation": "left"})
PCT = PercentFormatter(1, decimals=0)                              # axis labels as percentages
SEQ = sns.light_palette(C["terracotta"], as_cmap=True)   # one-hue sequential map for heatmaps

from scipy import stats
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

# -- Load the data -------------------------------------------------------------
# Needs two files: ev_m3_sample.csv (10,000 rows: the cleaned Module 2 sample)
#                  ev_m3_population.csv (50,000 rows: we treat these as "everybody")
def find(name):
    for folder in [".", "data", "../data"]:
        if os.path.exists(os.path.join(folder, name)):
            return os.path.join(folder, name)
    try:
        from google.colab import files
        print("Choose", name); files.upload(); return name
    except ImportError:
        raise FileNotFoundError(f"Put {name} next to this notebook or in a data folder.")

def add_derived(d):
    """The four derived columns built in Session 1 (pre-written so every notebook stands alone)."""
    d = d.copy()
    d["buy"] = (d["Will_Buy_EV"] == "Yes").astype(int)
    d["age_band"] = pd.cut(d["Age"], bins=[24, 34, 44, 54, 69], labels=["25-34", "35-44", "45-54", "55-69"])
    d["income_band"] = pd.qcut(d["Annual_Income_USD"], 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
    d["charger_total"] = d["Charging_Stations_Near_Home"] + d["Charging_Stations_Near_Work"]
    return d

df = add_derived(pd.read_csv(find("ev_m3_sample.csv")))        # the class sample
pop = add_derived(pd.read_csv(find("ev_m3_population.csv")))   # "everybody"
print("Sample:", len(df), "rows | Population:", len(pop), "rows")

## Part A. Hook: significant, but so what? (5 min)
**Predict first:** chargers near home vs buying. Will the p-value be small or large? Will the correlation be big or small?

In [ ]:
r, p = stats.____(df["Charging_Stations_Near_Home"], df["buy"])      # TODO: Pearson correlation with its p-value (pearsonr)
print(f"Correlation = {r:.3f}   p-value = {p:.4f}")

**YOUR ANSWER:** p is below 0.05, so it is 'significant'. Is a correlation of this size worth a marketing budget? Hold that thought for Part D.

> _write here_

## Part B. Confidence intervals: a range, not a point (10 min)

In [ ]:
lo, hi = proportion_confint(df["buy"].sum(), len(df), method="wilson")         # 95% interval for a rate
print(f"Yes rate = {df['buy'].____():.1%}; 95% interval = {lo:.1%} to {hi:.1%}")       # TODO: the Yes rate (mean)

**What does '95%' mean?** Let us *see* it. We draw **100 surveys of 500 people** from everybody, build an interval for each, and count how many capture the truth (red line). Intervals that miss are drawn in terracotta.
**Predict first:** how many of the 100 will capture the truth?

In [ ]:
P = pop["buy"].mean()                                           # the truth for everybody
rows = []
for i in range(100):
    s = pop["buy"].sample(500, random_state=i)
    lo, hi = proportion_confint(s.sum(), 500, method="wilson")
    rows.append((lo, hi, lo <= P <= hi))
cov = pd.DataFrame(rows, columns=["lo", "hi", "captured"])

fig, ax = plt.subplots(figsize=(7, 6))
for y, r in cov.iterrows():
    ax.plot([r.lo, r.hi], [y, y], color=C["lagoon"] if r.captured else C["terracotta"], lw=1.6)
ax.axvline(P, color=C["ink"], lw=1.2); ax.set_yticks([]); ax.xaxis.set_major_formatter(PCT)
ax.set_xlabel("95% interval for the Yes rate (each line = one survey of 500; black line = the truth)")
plt.tight_layout()
print(f"{cov['captured'].sum()} of 100 intervals captured the truth")

**YOUR ANSWER:** In your own words, what does '95% confidence' say about the *method*? Why is it wrong to say 'there is a 95% chance the truth is inside *this* interval'?

> _write here_

## Part C. Intervals for a gap (5 min)
A manager wants a range for *how much more buyers earn*, and for *how much higher rural buying is than urban*.

In [ ]:
yes = df[df["buy"] == 1]; no = df[df["buy"] == 0]
res = stats.ttest_ind(yes["Annual_Income_USD"], no["Annual_Income_USD"], equal_var=False)
ci = res.____()                                                 # TODO: 95% interval for the difference (confidence_interval)
gap = yes["Annual_Income_USD"].mean() - no["Annual_Income_USD"].mean()
print(f"Buyers earn {gap:,.0f} more; 95% interval {ci.low:,.0f} to {ci.high:,.0f}")

In [ ]:
def rate_gap_ci(a, b):
    """95% interval for the difference of two yes/no rates (a minus b), plain normal formula."""
    gap = a.mean() - b.mean()
    se = np.sqrt(a.mean() * (1 - a.mean()) / len(a) + b.mean() * (1 - b.mean()) / len(b))
    return gap, gap - 1.96 * se, gap + 1.96 * se

rural, urban = df[df["City_Type"] == "Rural"]["buy"], df[df["City_Type"] == "Urban"]["buy"]
gap, lo, hi = rate_gap_ci(rural, urban)
print(f"Class sample: rural minus urban = {gap:.1%}, 95% interval {lo:.1%} to {hi:.1%}")
true_gap = pop[pop["City_Type"] == "Rural"]["buy"].mean() - pop[pop["City_Type"] == "Urban"]["buy"].mean()
print(f"Everybody (50,000 rows): rural minus urban = {true_gap:.1%}")

**YOUR ANSWER:** Is the real (population) gap inside the class-sample interval? What does it say about quoting '5.2 points' in a report?

> _write here_

## Part D. Significant is not the same as important: effect size (5 min)
A p-value says *is there an effect*. An **effect size** says *how big*. **Cohen's d** measures the gap between two means in units of spread: about 0.2 small, 0.5 medium, 0.8 large. **Cramer's V** does the same for chi-square tables.

In [ ]:
def cohens_d(a, b):
    return (a.mean() - b.mean()) / np.sqrt((a.____() + b.____()) / 2)       # TODO: the spread measure for each group, squared: the variance (var)

rows = []
for col in ["Annual_Income_USD", "Daily_Commute_km", "Charging_Stations_Near_Home", "Age"]:
    rows.append({"column": col, "p_value": stats.ttest_ind(yes[col], no[col], equal_var=False).pvalue, "cohens_d": cohens_d(yes[col], no[col])})
pd.DataFrame(rows).round(4)

In [ ]:
table = pd.crosstab(df["City_Type"], df["buy"])
chi2, p, dof, exp = stats.chi2_contingency(table)
V = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))        # Cramer's V
print(f"City vs buying: p = {p:.1e}, Cramer's V = {V:.3f}")

**YOUR ANSWER:** Which columns have a tiny p-value *and* a tiny effect? Which have both a small p-value and a real-sized effect? Revisit your answer to Part A.

> _write here_

## Part E. Test many things and luck will find something (5 min)
We test **every column** against buying, then repeat with **20 columns of pure random numbers**.
**Predict first:** how many of the 20 random columns will look 'significant' at 0.05?

In [ ]:
pvals = {}
for col in ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned", "Charging_Stations_Near_Home", "Charging_Stations_Near_Work"]:
    pvals[col] = stats.ttest_ind(yes[col], no[col], equal_var=False).pvalue
for col in ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level", "Environmental_Concern_Level"]:
    pvals[col] = stats.chi2_contingency(pd.crosstab(df[col], df["buy"]))[1]

m = len(pvals)                                                  # number of tests
alpha_bonf = 0.05 / ____                                        # TODO: Bonferroni = alpha divided by the number of tests (m)
out = pd.DataFrame({"p": pd.Series(pvals)}).sort_values("p")
out["significant_at_0.05"] = out["p"] < 0.05
out["significant_after_Bonferroni"] = out["p"] < alpha_bonf
print(f"{m} tests; Bonferroni threshold = {alpha_bonf:.4f}")
out.round(4)

In [ ]:
rng = np.random.default_rng(2026)
noise_p = []
for i in range(20):
    x = rng.normal(size=len(df))                                # a column of pure noise
    noise_p.append(stats.ttest_ind(x[df["buy"] == 1], x[df["buy"] == 0], equal_var=False).pvalue)
print(f"{sum(q < 0.05 for q in noise_p)} of 20 noise columns look 'significant' at 0.05 (smallest p = {min(noise_p):.3f})")
print(f"After Bonferroni (threshold {0.05 / 20:.4f}): {sum(q < 0.05 / 20 for q in noise_p)}")

**YOUR ANSWER:** Which real column drops out after Bonferroni? If you tested 100 useless columns at 0.05, how many false alarms would you expect?

> _write here_

## Part F. Failing to find an effect is not proof of none: power (5 min)
The *real* rural-urban gap in everybody is about 3 points. **Predict first:** if we survey only 200 people, how often does a test detect it?

In [ ]:
def detect_rate(n, reps=200):
    """Share of surveys of size n in which the rural-urban gap is significant at 0.05."""
    hits = 0
    for i in range(reps):
        s = pop.sample(n, random_state=i)
        a, b = s.loc[s["City_Type"] == "Rural", "buy"], s.loc[s["City_Type"] == "Urban", "buy"]
        z, p = proportions_ztest([a.sum(), b.sum()], [len(a), len(b)])
        hits += p < 0.05
    return hits / reps

sizes = [200, 500, 1000, 2000, 5000]
power = [detect_rate(n) for n in sizes]
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(sizes, power, marker="o", color=C["lagoon"], lw=2)
ax.yaxis.set_major_formatter(PCT); ax.set_ylim(0, 1)
ax.set_xlabel("Survey size"); ax.set_ylabel("Share of surveys that detect the gap")
plt.tight_layout()
print(dict(zip(sizes, [f"{x:.0%}" for x in power])))

**YOUR ANSWER:** At n = 200 the gap is real but rarely detected. What would you tell a manager who says *'we ran a small survey and found no difference, so there is none'*?

> _write here_

## Part G. Project Charter v2 and team task (10 min)
Add a **hypothesis table** to your Module 2 charter. Fill one row per claim, using results from Sessions 2 to 4.

In [ ]:
charter_v2 = pd.DataFrame([
    {"claim": "Buyers earn more than non-buyers", "test": "____", "statistic": "____", "p": "____", "effect_and_interval": "____", "decision": "____"},
    {"claim": "City type relates to buying", "test": "____", "statistic": "____", "p": "____", "effect_and_interval": "____", "decision": "____"},
    {"claim": "A subsidy relates to buying", "test": "____", "statistic": "____", "p": "____", "effect_and_interval": "____", "decision": "____"},
    {"claim": "Chargers near home relate to buying", "test": "____", "statistic": "____", "p": "____", "effect_and_interval": "____", "decision": "____"},
])
charter_v2          # TODO: replace every "____" using your results from Parts C to E and Session 3

**Team task:** write a **five-sentence note to the marketing head**: what is solid, what is small, what is uncertain, and one action. Use plain English. No unexplained jargon.

**Checklist:** (1) names a decision; (2) gives a number *and* its interval; (3) separates significant from important; (4) mentions one lurking variable or caution (city and home charging); (5) admits the data is synthetic.

## Exit ticket (5 min)
1. p = 0.001 and Cohen's d = 0.05. Is this a finding worth acting on? Why or why not?
2. You test 40 columns at 0.05 and two come out significant. How worried should you be?
3. A small survey finds no difference between two stores. Can you say there is no difference?